# IQL en GPU: entrenamiento y búsqueda de hiperparámetros

Este notebook:
1. Hace una **búsqueda aleatoria de arquitectura/entrenamiento** (etapa 1) con τ=0,9, β=3, γ=0,9.
2. Hace la **grilla IQL de τ, β y γ** (etapa 2) sobre la mejor arquitectura, incluyendo γ=0 frente a γ>0.
3. Entrena **3 semillas** con la configuración elegida y evalúa **π** y **máximo de Q** en validación frente a las líneas base ajustadas con los mismos logs.
4. Deja **test reservado** (`RUN_TEST = False`).

Pensado para una VM con GPU NVIDIA A30 (24 GB), 128 cores y 396 GB de RAM. Usa `modelos/iql.py` con AMP (fp16) y TF32. La evaluación del protocolo es secuencial (un forward batch-1 por paso de sesión): el entrenamiento es rápido, la evaluación es el cuello.

Para una prueba rápida de punta a punta, poner `SMOKE = True`.

Las decisiones de diseño del modelo (codificadores separados, recompensa centrada, red objetivo, checkpoint de la última época, etc.) están documentadas en `modelos/iql.py` y en la guía del TP.

In [ ]:
SMOKE = False

DATASET = "netflix"
LOGS = "random"          # "random" | "mixto" | "experto"
HORIZON = 20
RUN_NAME = "iql_gpu_smoke" if SMOKE else f"iql_gpu_{LOGS}"

HPO_TRIALS = 1 if SMOKE else 8
HPO_PER_GROUP = 2 if SMOKE else 25
HPO_CONDITIONS = [(0, True), (0, False)]
MAX_BATCHES = 3 if SMOKE else None

STAGE1_IQL = dict(tau=0.9, beta=3.0, gamma=0.9)
FIXED_TRAIN = dict(group_dropout=0.3)
GRID_TAU = (0.9,) if SMOKE else (0.7, 0.9)
GRID_BETA = (3.0,) if SMOKE else (3.0, 10.0)
GRID_GAMMA = (0.9,) if SMOKE else (0.0, 0.9)

FINAL_SEEDS = (0,) if SMOKE else (0, 1, 2)
FINAL_PER_GROUP = 2 if SMOKE else 50
EVAL_SEEDS = (0,) if SMOKE else (0, 1, 2)

RUN_TEST = False

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "config_dataset.py").exists():
    raise RuntimeError("Ejecutar el notebook desde 2026/tp")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modelos.iql import IQLPolicy, train_iql
from protocolo.datos import load_test, load_train, make_episodes, num_items, split_by_user
from protocolo.evaluar import baseline_factories, evaluate, write_reports
from protocolo.logs import PRESETS, PreviousSystem, logged_episodes, logs_cache
from protocolo.metricas import summarize
from protocolo.simulador import SessionConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

torch.manual_seed(0)
np.random.seed(0)
print(f"torch {torch.__version__} | device: {DEVICE} | amp: {USE_AMP}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
if SMOKE:
    print("MODO SMOKE: corrida corta para validar el notebook")

In [ ]:
paths = Path(f"outputs/{RUN_NAME}")
paths.mkdir(parents=True, exist_ok=True)

train_all = load_train(DATASET)
train, val = split_by_user(train_all)
N_ITEMS = num_items(DATASET)

if LOGS == "random":
    train_ep = make_episodes(train, HORIZON)
    val_ep = make_episodes(val, HORIZON)
    baseline_train = train
else:
    logged_train = logged_episodes(
        LOGS, train, train, N_ITEMS, HORIZON, seed=0,
        cache=logs_cache(DATASET, LOGS, "train"),
    )
    logged_val = logged_episodes(
        LOGS, val, train, N_ITEMS, HORIZON, seed=1,
        cache=logs_cache(DATASET, LOGS, "val"),
    )
    train_ep, val_ep = logged_train.episodes, logged_val.episodes
    baseline_train = logged_train.logged_users()

print(f"logs: {LOGS} | catálogo: {N_ITEMS}")
print(f"episodios: train {train_ep.items.shape[0]:,} | val {val_ep.items.shape[0]:,}")
print("retorno medio de un episodio de train:", float(train_ep.ratings.sum(1).mean()))

In [ ]:
def stratified_subset(users, per_group, seed=0):
    rng = np.random.default_rng(seed)
    subset = []
    for group in range(8):
        members = [u for u in users if u.group == group]
        take = min(per_group, len(members))
        idx = rng.choice(len(members), size=take, replace=False)
        subset.extend(members[i] for i in idx)
    return subset


def eval_policy(policy, users, conditions, seeds):
    frames = []
    for warmup, visible in conditions:
        cfg = SessionConfig(horizon=HORIZON, warmup=warmup, group_visible=visible)
        df = evaluate(policy, users, N_ITEMS, cfg, seeds=seeds)
        df["warmup"] = warmup
        df["group_visible"] = visible
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def eval_iql(model, users, conditions, seeds, modes=("pi", "q")):
    frames = []
    for mode in modes:
        df = eval_policy(IQLPolicy(model, mode), users, conditions, seeds)
        df["mode"] = mode
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


hpo_users = stratified_subset(val, HPO_PER_GROUP, seed=0)
final_users = stratified_subset(val, FINAL_PER_GROUP, seed=1)
print(f"usuarios de val: HPO {len(hpo_users)} | final {len(final_users)}")

## 1. Búsqueda de hiperparámetros: arquitectura y entrenamiento

Búsqueda aleatoria sobre `d_model`, `n_layers`, `n_heads`, `dropout`, `lr`, `weight_decay`, `batch_size` y `epochs`, con τ=0,9, β=3, γ=0,9 fijos. `group_dropout` se fija en 0,3 para que el token "grupo desconocido" se entrene y la condición cold-start sea válida.

Cada trial se entrena una vez y se evalúa en un subconjunto estratificado de validación (25 usuarios por grupo) con el protocolo: warm-up 0, grupo visible y oculto, políticas π y máximo de Q. El score es el **normalizado medio** de las dos políticas y las dos condiciones. Las pérdidas se guardan sólo como diagnóstico.

En la A30 el entrenamiento es rápido; el tiempo lo domina la evaluación del protocolo. Si hace falta acortar, bajar `HPO_TRIALS` o `HPO_PER_GROUP`.

In [ ]:
rng = np.random.default_rng(123)


def sample_config():
    d_model = int(rng.choice([64, 96, 128]))
    head_options = [h for h in (4, 8) if d_model % h == 0]
    return {
        "d_model": d_model,
        "n_heads": int(rng.choice(head_options)),
        "n_layers": int(rng.choice([2, 3])),
        "dropout": float(rng.choice([0.0, 0.1, 0.2])),
        "lr": float(np.exp(rng.uniform(np.log(1e-4), np.log(1e-3)))),
        "weight_decay": float(rng.choice([1e-2, 1e-1])),
        "batch_size": int(rng.choice([256, 512, 1024])),
        "epochs": int(rng.choice([10, 15])),
    }


SMOKE_CONFIG = {
    "d_model": 32, "n_heads": 2, "n_layers": 1, "dropout": 0.0,
    "lr": 1e-3, "weight_decay": 1e-2, "batch_size": 32, "epochs": 1,
}

hpo_rows = []
for trial in range(HPO_TRIALS):
    cfg = SMOKE_CONFIG if SMOKE else sample_config()
    start = time.time()
    model, history = train_iql(
        train_ep, val_ep, N_ITEMS,
        seed=0,
        max_batches=MAX_BATCHES,
        device=DEVICE,
        amp=USE_AMP,
        output=paths / "hpo" / f"trial{trial:02d}",
        verbose=not SMOKE,
        **cfg, **STAGE1_IQL, **FIXED_TRAIN,
    )
    raw = eval_iql(model, hpo_users, HPO_CONDITIONS, seeds=(0,))
    summary = summarize(raw, by=["mode", "warmup", "group_visible"])
    norm = summary[("normalized", "mean")]
    pi_score = float(norm.xs("pi", level="mode").mean())
    q_score = float(norm.xs("q", level="mode").mean())
    score = (pi_score + q_score) / 2
    row = {
        **cfg, **STAGE1_IQL, **FIXED_TRAIN,
        "pi": pi_score, "q": q_score, "score": score,
        "val_pi": float(history.iloc[-1]["val_pi"]),
        "minutos": (time.time() - start) / 60,
    }
    hpo_rows.append(row)
    print(
        f"trial {trial}: score={score:.3f} | pi={pi_score:.3f} q={q_score:.3f} "
        f"| {row['minutos']:.1f} min | {cfg}"
    )
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

hpo = pd.DataFrame(hpo_rows).sort_values("score", ascending=False).reset_index(drop=True)
hpo.to_csv(paths / "hpo_resultados.csv", index=False)
hpo.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(np.arange(len(hpo)), hpo["score"], color="#4c72b0")
axes[0].set_xlabel("trial (ordenado por score)")
axes[0].set_ylabel("normalizado en val")
axes[0].set_title("HPO: score por trial")

axes[1].scatter(hpo["val_pi"], hpo["score"], s=40, color="#c44e52")
axes[1].set_xlabel("val π (pérdida AWR)")
axes[1].set_ylabel("normalizado en val")
axes[1].set_title("La pérdida de π no predice la calidad")
for idx, row in hpo.head(3).iterrows():
    axes[1].annotate(f"trial {idx}", (row["val_pi"], row["score"]), fontsize=7)
plt.tight_layout()
plt.show()

print("Mejor configuración de arquitectura:")
display(hpo.head(1).T)

## 2. Grilla τ, β y γ sobre la mejor arquitectura

Se entrena una vez por cada combinación de τ ∈ {0,7; 0,9}, β ∈ {3; 10} y γ ∈ {0; 0,9}, con la arquitectura ganadora y `group_dropout=0,3`. Para cada configuración se evalúan **π** y **máximo de Q** en validación (warm-up 0, visible/oculto).

La configuración final se elige por el **normalizado medio de ambas políticas y condiciones**, no por las pérdidas. La tabla y el gráfico por γ muestran qué parte del valor viene de la secuencia (γ=0 vs γ>0).

In [ ]:
best = hpo.iloc[0]
integer_keys = {"d_model", "n_heads", "n_layers", "batch_size", "epochs"}
best_model_cfg = {
    key: int(best[key]) if key in integer_keys else float(best[key])
    for key in ["d_model", "n_heads", "n_layers", "dropout", "lr",
                "weight_decay", "batch_size", "epochs"]
}
print("Arquitectura elegida:", best_model_cfg)

grid_rows = []
for tau in GRID_TAU:
    for beta in GRID_BETA:
        for gamma in GRID_GAMMA:
            iql_cfg = {"tau": float(tau), "beta": float(beta), "gamma": float(gamma)}
            start = time.time()
            model, history = train_iql(
                train_ep, val_ep, N_ITEMS,
                seed=0,
                max_batches=MAX_BATCHES,
                device=DEVICE,
                amp=USE_AMP,
                verbose=False,
                **best_model_cfg, **FIXED_TRAIN, **iql_cfg,
            )
            raw = eval_iql(model, hpo_users, HPO_CONDITIONS, seeds=(0,))
            summary = summarize(raw, by=["mode", "warmup", "group_visible"])
            norm = summary[("normalized", "mean")]
            pi_score = float(norm.xs("pi", level="mode").mean())
            q_score = float(norm.xs("q", level="mode").mean())
            grid_rows.append({
                **iql_cfg, "pi": pi_score, "q": q_score,
                "score": (pi_score + q_score) / 2,
                "val_pi": float(history.iloc[-1]["val_pi"]),
                "minutos": (time.time() - start) / 60,
            })
            print(
                f"tau={tau} beta={beta} gamma={gamma}: "
                f"pi={pi_score:.3f} q={q_score:.3f} ({grid_rows[-1]['minutos']:.1f} min)"
            )
            del model
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

grid = pd.DataFrame(grid_rows).sort_values("score", ascending=False).reset_index(drop=True)
grid.to_csv(paths / "grid_iql.csv", index=False)
grid.round(4)

In [ ]:
best_grid = grid.iloc[0]
best_iql_cfg = {
    "tau": float(best_grid["tau"]),
    "beta": float(best_grid["beta"]),
    "gamma": float(best_grid["gamma"]),
}
print("Configuración IQL elegida:", best_iql_cfg)

print("Grilla completa (normalizado medio):")
display(grid.round(3))

gamma_view = grid.groupby("gamma")[["pi", "q", "score"]].mean()
print("γ=0 frente a γ>0 (media sobre τ, β y condiciones):")
display(gamma_view.round(3))

fig, ax = plt.subplots(figsize=(5.5, 3.5))
gamma_view[["pi", "q"]].plot.bar(ax=ax)
ax.set_xlabel("γ")
ax.set_ylabel("normalizado en val")
ax.set_title("Contribución del valor futuro (π y máximo de Q)")
plt.tight_layout()
plt.show()

## 3. Entrenamiento final con 3 semillas

Se entrena la configuración elegida con 3 semillas. Cada semilla guarda `modelo_iql.pt`, `config.json` e `historial.csv` en `outputs/<RUN_NAME>/final/seed<k>/`. El checkpoint es el de la última época: en IQL las pérdidas de Q y π pueden subir mientras la política mejora.

In [ ]:
models, histories = {}, {}
for seed in FINAL_SEEDS:
    model, history = train_iql(
        train_ep, val_ep, N_ITEMS,
        seed=seed,
        max_batches=MAX_BATCHES,
        device=DEVICE,
        amp=USE_AMP,
        output=paths / "final" / f"seed{seed}",
        **best_model_cfg, **FIXED_TRAIN, **best_iql_cfg,
    )
    models[seed] = model
    histories[seed] = history

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for seed, history in histories.items():
    axes[0].plot(history["epoch"], history["train_v"], "--", alpha=0.4, label=f"V s{seed}")
    axes[0].plot(history["epoch"], history["train_q"], "-", alpha=0.8, label=f"Q s{seed}")
    axes[1].plot(history["epoch"], history["train_pi"], "--", alpha=0.5, label=f"train s{seed}")
    axes[1].plot(history["epoch"], history["val_pi"], "-", label=f"val s{seed}")
    axes[2].plot(history["epoch"], history["w_medio"], label=f"medio s{seed}")
    axes[2].plot(history["epoch"], history["w_truncado"], "--", label=f"truncado s{seed}")
axes[0].set_title("Pérdidas V y Q (train)")
axes[1].set_title("π ponderada (AWR)")
axes[2].set_title("Pesos AWR")
for axis in axes:
    axis.set_xlabel("época")
    axis.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 4. Evaluación en validación

Se evalúan π y máximo de Q en las 4 condiciones (warm-up 0/5 × grupo visible/oculto) con las 3 semillas de entrenamiento y 3 semillas de evaluación. Se reportan media, IC 95 % por usuario, hit rate y NDCG@10, más el desvío entre semillas de entrenamiento (separado del IC). También se evalúan las líneas base ajustadas con los mismos logs y, si `LOGS != "random"`, el sistema anterior con ε=0.

In [ ]:
conditions = [(0, True), (0, False), (5, True), (5, False)]

frames = []
for seed, model in models.items():
    df = eval_iql(model, final_users, conditions, EVAL_SEEDS)
    df["train_seed"] = seed
    frames.append(df)
val_raw = pd.concat(frames, ignore_index=True)
val_raw.to_csv(paths / "iql_val_sesiones.csv", index=False)

val_summary = summarize(val_raw, by=["mode", "warmup", "group_visible"])
norm = val_summary[("normalized", "mean")]
table = pd.concat({
    "normalized": norm,
    "ci95": val_summary[("normalized", "ci95")],
    "hit_rate": val_summary[("hit_rate", "mean")],
    "ndcg@10": val_summary[("ndcg@10", "mean")],
}, axis=1)
print("Validación: media e IC 95 % por usuario")
display(table.round(3))

seed_stats = (
    val_raw.groupby(["mode", "warmup", "group_visible", "train_seed"])["normalized"].mean()
    .groupby(["mode", "warmup", "group_visible"]).agg(["mean", "std"]).round(3)
)
print("Media y desvío entre semillas de entrenamiento")
display(seed_stats)

In [ ]:
factories = baseline_factories(baseline_train, N_ITEMS)
if LOGS != "random":
    _, temperature = PRESETS[LOGS]
    factories["sistema_anterior_eps0"] = lambda: PreviousSystem.fit(
        train, N_ITEMS, 0.0, temperature, np.random.default_rng(0)
    )

base_frames = []
for name, factory in factories.items():
    df = eval_policy(factory(), final_users, conditions, EVAL_SEEDS)
    df["policy"] = name
    base_frames.append(df)
base_raw = pd.concat(base_frames, ignore_index=True)
base_summary = summarize(base_raw, by=["policy", "warmup", "group_visible"])
base_norm = base_summary[("normalized", "mean")]

comparison = base_norm.unstack(["warmup", "group_visible"])
for mode in ("pi", "q"):
    comparison.loc[f"iql_{mode}"] = norm.xs(mode, level="mode")
print("Comparación en validación (normalizado, 0=azar, 1=oráculo):")
display(comparison.round(3))

In [ ]:
per_group = summarize(
    val_raw, by=["mode", "warmup", "group_visible", "group"]
)[("normalized", "mean")]
per_group_table = per_group.unstack("group")
print("Resultados por grupo (normalizado):")
display(per_group_table.round(3))

## 5. Test reservado

`RUN_TEST = False` por defecto: test se usa **una sola vez**, cuando la configuración, γ, τ, β y las semillas ya están congelados en validación. Al ponerlo en `True`, evalúa las 3 semillas de entrenamiento (π y máximo de Q) y las líneas base con los mismos logs, y escribe `sesiones.csv`, `resumen.csv` y `resumen_por_grupo.csv`.

In [ ]:
if RUN_TEST:
    test_users = load_test(DATASET)
    test_frames = []
    for seed, model in models.items():
        df = eval_iql(model, test_users, conditions, EVAL_SEEDS)
        df["policy"] = df["mode"].map(lambda mode: f"iql_{mode}_seed{seed}")
        test_frames.append(df)
    for name, factory in factories.items():
        df = eval_policy(factory(), test_users, conditions, EVAL_SEEDS)
        df["policy"] = name
        test_frames.append(df)
    test_raw = pd.concat(test_frames, ignore_index=True)
    write_reports(test_raw, paths / "eval" / "test")
else:
    print("RUN_TEST=False: test reservado. Cambiarlo a True una sola vez, al final.")

## Checklist antes de la entrega

- [ ] La arquitectura y τ, β, γ se eligieron **sólo con validación**.
- [ ] Se reportan 3 semillas de entrenamiento: media y desvío entre semillas, separados del IC 95 % por usuario.
- [ ] Las líneas base se ajustaron con los **mismos logs** que el modelo.
- [ ] Están las dos políticas: π (AWR) y máximo de Q, con la comparación γ=0 vs γ>0.
- [ ] Están los resultados por grupo y las métricas secundarias (hit rate, NDCG@10).
- [ ] Si se usa `mixto`, se repitió todo con `LOGS = "mixto"` y se reajustaron las líneas base con esos logs.
- [ ] `RUN_TEST = True` se usó una sola vez, al final.